<a href="https://colab.research.google.com/github/fardinAhmed285/tsp-crossover-study/blob/main/tsp_crossover_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import urllib.request

# Download ulysses22.tsp from a reliable GitHub repository
url = "https://raw.githubusercontent.com/mastqe/tsplib/master/ulysses22.tsp"
urllib.request.urlretrieve(url, "ulysses22.tsp")

print("Downloaded ulysses22.tsp successfully!")

Downloaded ulysses22.tsp successfully!


In [2]:
!pip install deap tsplib95

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 37.1 MB/s  0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 2.5.1
    Uninstalling wrapt-2.5.1:
      Successfully uninstalled wrapt-2.5.1
  Attempting uninstall: tabulate
    Found existing installation: tabulate 0.9.0
    Uninstalling tabulate-0.9.0:
      Successfully uninstalled tabulate-0.9.0
  Attempting uninstall: networkx
    Found existing installation: networkx 3.7
    Uninstalling networkx-3.7:
      Successfully uninstalled networkx-3.7
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [deap]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
momepy 0.11.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
spopt 0.7.0 requires networkx>=3.2, but you have networkx 2.8.8 whi

In [9]:
import random
import numpy as np
from deap import base, creator, tools, algorithms
import tsplib95

# 1. Load the ulysses22 problem file
problem = tsplib95.load("ulysses22.tsp")
original_nodes = list(problem.get_nodes())
n_cities = len(original_nodes)

# 2. Set up DEAP framework safely (prevents duplicate class errors if re-run)
if "FitnessMin" not in creator.__dict__:
    creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
if "Individual" not in creator.__dict__:
    creator.create("Individual", list, fitness=creator.FitnessMin)

# Fitness evaluation function
def eval_tsp(individual):
    total_distance = 0
    for i in range(len(individual)):
        u = original_nodes[individual[i]]
        v = original_nodes[individual[(i + 1) % len(individual)]]
        total_distance += problem.get_weight(u, v)
    return (total_distance,)

# Helper function to extract edges
def get_edges(route):
    edges = set()
    n = len(route)
    for i in range(n):
        u = route[i]
        v = route[(i + 1) % n]
        edges.add(tuple(sorted((u, v))))
    return edges

# 3. Experiment Runner Function
def run_experiment(crossover_func, operator_name, ngen=50):
    toolbox = base.Toolbox()
    toolbox.register("indices", random.sample, range(n_cities), n_cities)
    toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.indices)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", eval_tsp)

    heritability_log = []

    def custom_mate(ind1, ind2):
        p1_copy = list(ind1)
        p2_copy = list(ind2)

        # Apply the chosen crossover operator
        crossover_func(ind1, ind2)

        # Calculate edge heritability
        p1_edges = get_edges(p1_copy)
        p2_edges = get_edges(p2_copy)
        parent_pool = p1_edges.union(p2_edges)

        for child in [ind1, ind2]:
            child_edges = get_edges(child)
            inherited = sum(1 for edge in child_edges if edge in parent_pool)
            heritability_log.append(inherited / len(child_edges))

        return ind1, ind2

    toolbox.register("mate", custom_mate)
    toolbox.register("mutate", tools.mutShuffleIndexes, indpb=0.05)
    toolbox.register("select", tools.selTournament, tournsize=3)

    # Run GA execution
    random.seed(42) # Keep seed identical for a fair comparison
    pop = toolbox.population(n=100)
    hof = tools.HallOfFame(1)

    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("min", np.min)

    print(f"Running GA with {operator_name}...")
    pop, logbook = algorithms.eaSimple(pop, toolbox, cxpb=0.9, mutpb=0.1, ngen=ngen,
                                       stats=stats, halloffame=hof, verbose=False)

    avg_heritability = np.mean(heritability_log) * 100
    best_distance = hof[0].fitness.values[0]

    return best_distance, avg_heritability

# 4. Execute Both Experiments and Print Comparison Table
if __name__ == "__main__":
    print("=== Starting TSP Crossover Comparison Study ===")

    # Run Order Crossover (OX)
    ox_dist, ox_herit = run_experiment(tools.cxOrdered, "Order Crossover (OX)", ngen=50)

    # Run Partially Mapped Crossover (PMX)
    pmx_dist, pmx_herit = run_experiment(tools.cxPartialyMatched, "Partially Mapped Crossover (PMX)", ngen=50)

    # Print Final Summary Table for your report
    print("\n" + "="*57)
    print("            FINAL OPERATOR COMPARISON RESULTS          ")
    print("="*57)
    print(f"{'Operator Name':<28} | {'Best Distance':<13} | {'Heritability':<12}")
    print("-"*57)
    print(f"{'Order Crossover (OX)':<28} | {ox_dist:<13.1f} | {ox_herit:.2f}%")
    print(f"{'Partially Mapped (PMX)':<28} | {pmx_dist:<13.1f} | {pmx_herit:.2f}%")
    print("="*57)
    print("Optimal Known Distance for ulysses22: 7013")

=== Starting TSP Crossover Comparison Study ===
Running GA with Order Crossover (OX)...
Running GA with Partially Mapped Crossover (PMX)...

            FINAL OPERATOR COMPARISON RESULTS          
Operator Name                | Best Distance | Heritability
---------------------------------------------------------
Order Crossover (OX)         | 7882.0        | 87.06%
Partially Mapped (PMX)       | 8386.0        | 90.13%
Optimal Known Distance for ulysses22: 7013
